In [ ]:
import pandas as pd

df = pd.read_csv("/content/enron_spam_data.zip")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset Shape: (33716, 5)

Columns:
['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date']


,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB

Missing Values:
Message ID      0
Subject       289
Message       371
Spam/Ham        0
Date            0
dtype: int64

Duplicate Rows: 0

First 5 Rows:


,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [ ]:
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(10))


Message ID:
Message ID
33715    1
0        1
1        1
2        1
33699    1
33698    1
33697    1
33696    1
33695    1
33694    1
Name: count, dtype: int64

Subject:
Subject
NaN                                                       289
schedule crawler : hourahead failure                      185
re :                                                      162
all graphics software available , cheap oem versions .    107
localized software , all languages available .            101
perfect logo charset = koi 8 - r " >                       97
enron mentions                                             95
are you ready to get it ?                                  92
save your money buy getting this thing here                87
failure notice                                             71
Name: count, dtype: int64

Message:
Message
NaN                                                                                                                                                           

In [ ]:

import pandas as pd
import numpy as np

# Work on a copy
data = df.copy()

# Standardize labels
data["Spam/Ham"] = data["Spam/Ham"].astype(str).str.strip().str.lower()

# Keep only valid target labels
data = data[data["Spam/Ham"].isin(["spam", "ham"])].copy()

# Handle missing subject/body individually
data["Subject"] = data["Subject"].fillna("").astype(str).str.strip()
data["Message"] = data["Message"].fillna("").astype(str).str.strip()

# Combine subject and email body
data["text"] = (
    "Subject: " + data["Subject"] +
    "\n\nMessage: " + data["Message"]
).str.strip()

# Remove records with no usable email content
data = data[data["text"].str.replace(
    r"Subject:\s*Message:\s*", "", regex=True
).str.strip().ne("")].copy()

# Normalize text for duplicate detection only
data["duplicate_key"] = (
    data["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

before = len(data)
data = data.drop_duplicates(subset="duplicate_key").copy()
removed = before - len(data)

# Final dataset
data = data[["text", "Spam/Ham", "Date"]].reset_index(drop=True)

print("Final dataset shape:", data.shape)
print("Duplicate emails removed:", removed)
print("\nClass distribution:")
print(data["Spam/Ham"].value_counts())
print("\nMissing values:")
print(data.isnull().sum())

display(data.head(3))


Final dataset shape: (30138, 3)
Duplicate emails removed: 3527

Class distribution:
Spam/Ham
ham     15590
spam    14548
Name: count, dtype: int64

Missing values:
text        0
Spam/Ham    0
Date        0
dtype: int64


,text,Spam/Ham,Date
0,Subject: christmas tree farm pictures\n\nMessage:,ham,1999-12-10
1,"Subject: vastar resources , inc .\n\nMessage: ...",ham,1999-12-13
2,Subject: calpine daily gas nomination\n\nMessa...,ham,1999-12-14


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Stratified split keeps spam/ham proportions similar
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["Spam/Ham"],
    test_size=0.20,
    random_state=42,
    stratify=data["Spam/Ham"]
)

models = {
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=200_000,
            sublinear_tf=True
        )),
        ("classifier", MultinomialNB(alpha=0.1))
    ]),

    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=200_000,
            sublinear_tf=True
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining: {name}")
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Spam Precision": precision_score(
            y_test, predictions, pos_label="spam"
        ),
        "Spam Recall": recall_score(
            y_test, predictions, pos_label="spam"
        ),
        "Spam F1": f1_score(
            y_test, predictions, pos_label="spam"
        )
    })

    print(classification_report(
        y_test, predictions, labels=["ham", "spam"], zero_division=0
    ))

    print("Confusion matrix [ham, spam]:")
    print(confusion_matrix(y_test, predictions, labels=["ham", "spam"]))

results_df = pd.DataFrame(results).sort_values(
    by="Spam F1", ascending=False
)

display(results_df.round(4))



Training: Naive Bayes
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      3118
        spam       0.99      0.99      0.99      2910

    accuracy                           0.99      6028
   macro avg       0.99      0.99      0.99      6028
weighted avg       0.99      0.99      0.99      6028

Confusion matrix [ham, spam]:
[[3095   23]
 [  33 2877]]

Training: Logistic Regression
              precision    recall  f1-score   support

         ham       1.00      0.98      0.99      3118
        spam       0.98      1.00      0.99      2910

    accuracy                           0.99      6028
   macro avg       0.99      0.99      0.99      6028
weighted avg       0.99      0.99      0.99      6028

Confusion matrix [ham, spam]:
[[3063   55]
 [   9 2901]]


,Model,Accuracy,Spam Precision,Spam Recall,Spam F1
0,Naive Bayes,0.9907,0.9921,0.9887,0.9904
1,Logistic Regression,0.9894,0.9814,0.9969,0.9891


In [ ]:

# Check which dataset variables are currently available
print("data exists:", "data" in globals())
print("df exists:", "df" in globals())
print("final_data exists:", "final_data" in globals())


data exists: True
df exists: True
final_data exists: False


In [ ]:

# Re-create final_data from the cleaned dataset
final_data = data[["text", "Spam/Ham", "Date"]].copy()

print(final_data.shape)
display(final_data.head())


(30138, 3)


,text,Spam/Ham,Date
0,Subject: christmas tree farm pictures\n\nMessage:,ham,1999-12-10
1,"Subject: vastar resources , inc .\n\nMessage: ...",ham,1999-12-13
2,Subject: calpine daily gas nomination\n\nMessa...,ham,1999-12-14
3,Subject: re : issue\n\nMessage: fyi - see note...,ham,1999-12-14
4,Subject: meter 7268 nov allocation\n\nMessage:...,ham,1999-12-14


In [ ]:

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# 1. Prepare data
temporal_data = final_data.copy()
temporal_data["Date"] = pd.to_datetime(
    temporal_data["Date"], errors="coerce"
)
temporal_data = temporal_data.dropna(
    subset=["Date", "text", "Spam/Ham"]
)
temporal_data = temporal_data.sort_values("Date").reset_index(drop=True)

# 2. Split by date: oldest 80% of unique dates for training
unique_dates = sorted(temporal_data["Date"].dt.date.unique())
cutoff_index = max(1, int(len(unique_dates) * 0.80))
cutoff_index = min(cutoff_index, len(unique_dates) - 1)
cutoff_date = unique_dates[cutoff_index]

train = temporal_data[temporal_data["Date"].dt.date < cutoff_date]
test = temporal_data[temporal_data["Date"].dt.date >= cutoff_date]

print("Cutoff date:", cutoff_date)
print("Training emails:", len(train))
print("Testing emails:", len(test))
print("\nTraining labels:\n", train["Spam/Ham"].value_counts())
print("\nTesting labels:\n", test["Spam/Ham"].value_counts())

# 3. Compare models
models = {
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=2,
            max_features=200_000, sublinear_tf=True
        )),
        ("classifier", MultinomialNB(alpha=0.1))
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=2,
            max_features=200_000, sublinear_tf=True
        )),
        ("classifier", LogisticRegression(
            max_iter=1000, class_weight="balanced"
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(train["text"], train["Spam/Ham"])
    pred = model.predict(test["text"])

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(test["Spam/Ham"], pred),
        "Spam Precision": precision_score(
            test["Spam/Ham"], pred, pos_label="spam", zero_division=0
        ),
        "Spam Recall": recall_score(
            test["Spam/Ham"], pred, pos_label="spam", zero_division=0
        ),
        "Spam F1": f1_score(
            test["Spam/Ham"], pred, pos_label="spam", zero_division=0
        )
    })

    print(f"\n{name}")
    print(classification_report(
        test["Spam/Ham"], pred,
        labels=["ham", "spam"], zero_division=0
    ))
    print("Confusion matrix [ham, spam]:")
    print(confusion_matrix(
        test["Spam/Ham"], pred, labels=["ham", "spam"]
    ))

results_df = pd.DataFrame(results).sort_values(
    "Spam F1", ascending=False
)
display(results_df.round(4))


Cutoff date: 2004-11-05
Training emails: 20584
Testing emails: 9554

Training labels:
 Spam/Ham
ham     15590
spam     4994
Name: count, dtype: int64

Testing labels:
 Spam/Ham
spam    9554
Name: count, dtype: int64

Naive Bayes
              precision    recall  f1-score   support

         ham       0.00      0.00      0.00         0
        spam       1.00      0.96      0.98      9554

    accuracy                           0.96      9554
   macro avg       0.50      0.48      0.49      9554
weighted avg       1.00      0.96      0.98      9554

Confusion matrix [ham, spam]:
[[   0    0]
 [ 425 9129]]

Logistic Regression
              precision    recall  f1-score   support

         ham       0.00      0.00      0.00         0
        spam       1.00      0.97      0.99      9554

    accuracy                           0.97      9554
   macro avg       0.50      0.49      0.49      9554
weighted avg       1.00      0.97      0.99      9554

Confusion matrix [ham, spam]:
[[   0   

,Model,Accuracy,Spam Precision,Spam Recall,Spam F1
1,Logistic Regression,0.9729,1.0,0.9729,0.9863
0,Naive Bayes,0.9555,1.0,0.9555,0.9773


In [ ]:

# Check how spam and ham are distributed over time
temporal_data["YearMonth"] = temporal_data["Date"].dt.to_period("M")

monthly_counts = pd.crosstab(
    temporal_data["YearMonth"],
    temporal_data["Spam/Ham"]
)

display(monthly_counts.tail(24))

print("\nSpam/Ham distribution by year:")
temporal_data["Year"] = temporal_data["Date"].dt.year

display(pd.crosstab(
    temporal_data["Year"],
    temporal_data["Spam/Ham"]
))



Spam/Ham,ham,spam
YearMonth,,
2003-07,0,2
2003-11,0,1
2003-12,0,98
2004-01,0,211
2004-02,0,206
2004-03,0,209
2004-04,0,277
2004-05,0,282
2004-06,0,253



Spam/Ham distribution by year:


Spam/Ham,ham,spam
Year,,
1999,136,0
2000,6527,0
2001,7659,30
2002,1268,943
2003,0,103
2004,0,5505
2005,0,7967


In [ ]:

# Exact label distribution by year
check = final_data.copy()

check["Date"] = pd.to_datetime(check["Date"], errors="coerce")
check["Year"] = check["Date"].dt.year

print(pd.crosstab(check["Year"], check["Spam/Ham"], margins=True))

# Check if the same message content appears under both labels
check["normalized_text"] = (
    check["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

label_conflicts = (
    check.groupby("normalized_text")["Spam/Ham"]
    .nunique()
    .gt(1)
    .sum()
)

print("\nUnique email texts with conflicting labels:", label_conflicts)


Spam/Ham    ham   spam    All
Year                         
1999        136      0    136
2000       6527      0   6527
2001       7659     30   7689
2002       1268    943   2211
2003          0    103    103
2004          0   5505   5505
2005          0   7967   7967
All       15590  14548  30138

Unique email texts with conflicting labels: 0


In [ ]:

import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Use the cleaned dataset
X = final_data["text"].fillna("").astype(str)
y = final_data["Spam/Ham"].astype(str).str.strip().str.lower()

# Final pipeline: text -> TF-IDF -> Naive Bayes
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=200_000,
        sublinear_tf=True
    )),
    ("classifier", MultinomialNB(alpha=0.1))
])

# Train on all cleaned records
final_model.fit(X, y)

# Save model
model_path = Path("/content/spam_classifier.pkl")
joblib.dump(final_model, model_path)

print("Model saved at:", model_path)
print("Training records:", len(X))
print("Model classes:", final_model.classes_)

# Quick smoke test
samples = [
    "Congratulations! You won a free prize. Click here now!",
    "Hi, can we schedule the project meeting for tomorrow?"
]

for message in samples:
    prediction = final_model.predict([message])[0]
    print(f"\nEmail: {message}")
    print("Prediction:", prediction.upper())



Model saved at: /content/spam_classifier.pkl
Training records: 30138
Model classes: ['ham' 'spam']

Email: Congratulations! You won a free prize. Click here now!
Prediction: SPAM

Email: Hi, can we schedule the project meeting for tomorrow?
Prediction: HAM


In [ ]:

from google.colab import files

files.download("/content/spam_classifier.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1
